<a href="https://colab.research.google.com/github/alexander-toschev/cv-course/blob/main/invariant/Practise_Invariant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 📆 Colab: Обучение CNN с аугментациями на CIFAR-10

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

# Аугментации
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
])

# Датасеты
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# Простая модель
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Обучение
net = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
net.to(device)

for epoch in range(10):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:
            print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Finished Training')

# Тестирование
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f}%')

100%|██████████| 170M/170M [00:13<00:00, 12.6MB/s]


[1, 100] loss: 2.032
[1, 200] loss: 1.773
[1, 300] loss: 1.631
[2, 100] loss: 1.495
[2, 200] loss: 1.459
[2, 300] loss: 1.401
[3, 100] loss: 1.292
[3, 200] loss: 1.270
[3, 300] loss: 1.242
[4, 100] loss: 1.188
[4, 200] loss: 1.176
[4, 300] loss: 1.158
[5, 100] loss: 1.122
[5, 200] loss: 1.092
[5, 300] loss: 1.100
[6, 100] loss: 1.066
[6, 200] loss: 1.040
[6, 300] loss: 1.036
[7, 100] loss: 1.018
[7, 200] loss: 1.029
[7, 300] loss: 0.989
[8, 100] loss: 0.975
[8, 200] loss: 0.959
[8, 300] loss: 0.956
[9, 100] loss: 0.959
[9, 200] loss: 0.929
[9, 300] loss: 0.927
[10, 100] loss: 0.904
[10, 200] loss: 0.911
[10, 300] loss: 0.897
Finished Training
Accuracy of the network on the 10000 test images: 72.41%
